In [1]:
import glob
import os
import sys 
import numpy as np

import torch
import torchvision
from tqdm import tqdm
from torch import nn
from torch.autograd import Variable
from torch.utils.data import DataLoader
from torchvision import transforms
from torchvision.utils import save_image
#from thop import profile
import torch.nn.functional as F
import os
import gc
import matplotlib.pyplot as plt
%matplotlib inline

import torch.utils.data as data
from PIL import Image
import numpy as np
from torchvision.utils import save_image
import torch
import torch.nn.init as init
#from metrics import jaccard_index, f1_score, LogNLLLoss,classwise_f1
#from utils import chk_mkdir, Logger, MetricList
import cv2
#from functools import partial
#from random import randint
import timeit
#import thop
from torch.cuda.amp import autocast, GradScaler
#from lib.utils import adjust_learning_rate 
import matplotlib.pyplot as plt

In [2]:
#pip install torch

In [3]:
#pip install import-ipynb

In [9]:
import import_ipynb
import utils_gray_2
#import axialnet

In [10]:
#sys.path.insert(0, r'E:\workplace\teases_implementation')
#import utils_gray 

#import myadd

In [11]:
'''import import_ipynb
import myadd
import utils_gray'''

'import import_ipynb\nimport myadd\nimport utils_gray'

In [37]:
#from utils_gray_2 import RandomFlip, RandomRotate, RandomContrast,ImageToImage3D_SmallScale
from utils_gray_2 import ImageToImage3D_SmallScale
from axialnet import gated_3d
imgchant = 4

In [14]:
path = r"E:\workplace\teases_implementation\dataset"

In [15]:
'''import nibabel as nib

nii = nib.load(os.path.join(self.input_path, 'BraTS-GLI-00012-000-t2f.nii'))

image = nii.get_fdata().astype(np.float32)
image = nii.get_fdata().astype(np.int64)'''


"import nibabel as nib\n\nnii = nib.load(os.path.join(self.input_path, 'BraTS-GLI-00012-000-t2f.nii'))\n\nimage = nii.get_fdata().astype(np.float32)\nimage = nii.get_fdata().astype(np.int64)"

In [16]:
'''t1ce_list = sorted(glob.glob(fr'{path}/*/*t1c.nii'))
t1n_list = sorted(glob.glob(fr'{path}/*/*t1n.nii'))
t2_flair_list = sorted(glob.glob(fr'{path}/*/*t2f.nii'))
t2w_list = sorted(glob.glob(fr'{path}/*/*t2w.nii'))
mask_list = sorted(glob.glob(fr'{path}/*/*seg.nii'))'''

"t1ce_list = sorted(glob.glob(fr'{path}/*/*t1c.nii'))\nt1n_list = sorted(glob.glob(fr'{path}/*/*t1n.nii'))\nt2_flair_list = sorted(glob.glob(fr'{path}/*/*t2f.nii'))\nt2w_list = sorted(glob.glob(fr'{path}/*/*t2w.nii'))\nmask_list = sorted(glob.glob(fr'{path}/*/*seg.nii'))"

In [17]:
'''print(len(t1ce_list))
print(len(t1n_list))
print(len(t2_flair_list))
print(len(t2w_list))
print(len(mask_list))'''

'print(len(t1ce_list))\nprint(len(t1n_list))\nprint(len(t2_flair_list))\nprint(len(t2w_list))\nprint(len(mask_list))'

In [18]:
#print(t1ce_list)

In [19]:
modelname = "gated3d"
imgsize = 224
imgdepth = 160

'''if args.crop is not None:
    crop = (args.crop, args.crop)
else:
    crop = None'''

'if args.crop is not None:\n    crop = (args.crop, args.crop)\nelse:\n    crop = None'

In [27]:
def show_image_mask(my_image,my_mask, name1,name2 , n=0):
    import matplotlib.pyplot as plt
    %matplotlib inline
    
    # نمایش تصویر اول
    plt.subplot(1, 2, 1)  # 1 سطر، 2 ستون، موقعیت 1
    plt.imshow(my_image[0,n,:,:,50],cmap='gray')
    plt.title(f'{name1} {tuple(my_image.shape)}')
    plt.axis('on')
        
    # نمایش تصویر دوم
    plt.subplot(1, 2, 2)  # 1 سطر، 2 ستون، موقعیت 2
    plt.imshow(my_mask[0,:,:,50],cmap='gray')
    plt.title(f'{name2} {tuple(my_mask.shape)}' , fontsize=10)
    plt.axis('on')
        
    plt.tight_layout()
    plt.show()

In [28]:
device = torch.device("cuda")
seed = 3407
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed(seed) 
best_val=0
best_loss=99


In [34]:
#criterion = LogNLLLoss()

a = 0
tem = 1
kfold=5
batch_size = 1

for i in range(kfold):
    print('*' * 25, 'Num', i + 1, 'fold', '*' * 25)
    
    #train_dataset = ImageToImage3D_SmallScale(Training = True ,shape = (imgdepth,imgsize,imgsize), kfold = kfold, i = i, dataset_path=path,  RandomFlip = rf_train,RandomRotate = rr_train,RandomContrast = rc_train )
    #val_dataset = ImageToImage3D_SmallScale(Training = False ,shape = (imgdepth,imgsize,imgsize), kfold = kfold, i = i, dataset_path=path,  RandomFlip = rf_train,RandomRotate = rr_train,RandomContrast = rc_train)
    #dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    #my_image ,my_mask, my_name = next(iter(dataloader))

    
    train_dataset = ImageToImage3D_SmallScale(Training = True ,shape = ( imgsize,imgsize , imgdepth), kfold = kfold, i =1, dataset_path=path )
    dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    
    image_ori, mask_ori, adjusted_image, adjusted_mask, image_aug, mask_aug , my_name = next(iter(dataloader))

    #show_image_mask(image_ori ,mask_ori,'image_ori','mask_ori')
    #show_image_mask(adjusted_image ,adjusted_mask, 'adjusted_image','adjusted_mask')
    #show_image_mask(image_aug ,mask_aug,'image_aug','mask_ori_aug',0)
    #show_image_mask(image_aug ,mask_aug,'image_aug','mask_ori_aug',1)
    #show_image_mask(image_aug ,mask_aug,'image_aug','mask_ori_aug',2)
    #show_image_mask(image_aug ,mask_aug,'image_aug','mask_ori_aug',3)
    
    '''for batch in dataloader:
        images, masks, filenames = batch
        print("Batch Image Shape:", images.shape)
        print("Batch Mask Shape:", masks.shape)
        print("Filenames:", filenames)
    
    print('55555555555555555555555555')

    break
    '''
    

    
    if(kfold>1):
        valloader = DataLoader(val_dataset, batch_size=1, shuffle=True)
    else:
        valloader = DataLoader(train_dataset, batch_size=1, shuffle=False)

    train_loss_sum, valid_loss_sum = 0, 0
    train_dice_sum , valid_dice_sum = 0, 0
    
    if modelname == "gated3d":
        model = gated_3d(img_size=imgsize, img_depth = imgdepth, imgchan=imgchant)
    '''
    
     if torch.cuda.device_count() > 1:
        print("Let's use", torch.cuda.device_count(), "GPUs!")
        # dim = 0 [30, xxx] -> [10, ...], [10, ...], [10, ...] on 3 GPUs
        model = nn.DataParallel(model, device_ids=[0, 1]).cuda()
        model.to(device)
        optimizer = torch.optim.AdamW(list(model.parameters()), lr=args.learning_rate,
                                 weight_decay=1e-5, betas=(0.9, 0.999))

    for epoch in range(args.epochs):
        pass
    '''

************************* Num 1 fold *************************
len(images_list_all) :   3
*****************path is : E:\workplace\teases_implementation\dataset\BraTS-GLI-00009-000\BraTS-GLI-00009-000-seg.nii
image_ori.shape:    (4, 240, 240, 155)
mask_ori.shape:    (240, 240, 155)
adjusted_image.shape:    (4, 224, 224, 160)
adjusted_mask.shape:    (224, 224, 160)
[Flip] key=image, axis=3, shape=(4, 224, 224, 160)
[Flip] key=label, axis=3, shape=(224, 224, 160)
[Rotate] axis=(0, 2), angle=8.29
*****************path is : E:\workplace\teases_implementation\dataset\BraTS-GLI-00006-000\BraTS-GLI-00006-000-seg.nii
image_ori.shape:    (4, 240, 240, 155)
mask_ori.shape:    (240, 240, 155)
adjusted_image.shape:    (4, 224, 224, 160)
adjusted_mask.shape:    (224, 224, 160)
[Flip] key=image, axis=1, shape=(4, 224, 224, 160)
[Flip] key=image, axis=2, shape=(4, 224, 224, 160)
[Flip] key=label, axis=1, shape=(224, 224, 160)
[Flip] key=label, axis=2, shape=(224, 224, 160)
[Rotate] axis=(0, 2), angle=

ValueError: too many values to unpack (expected 3)

In [ ]:
dataset_path = path
input_path_image = os.path.join(dataset_path, 'images')
input_path_mask = os.path.join(dataset_path, 'masks')
output_path = os.path.join(dataset_path, '')
images_list_all = os.listdir(input_path_image)

#images_list_all